In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.config import RunnableConfig
from langchain.tools import tool
from typing import Any, cast
load_dotenv()

os.environ["GROQ_API_KEY"]
print("API Key loaded")

## Intro to Guardrails in Langchain

Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

+ Before the agent starts (input guardrails)
+ After it completes (output guardrails)
+ Around model and tool calls

#### Common Use Cases:
| Use Case | Example |
|---|---|
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |

## Two Approaches to Guardrails

#### Deterministic Guardrails
+ Rule-based: regex, keyword matching, explicit checks
+ Fast, predictable, cost-effective
+ May miss nuanced violations

#### Model-Based Guardrails
+ Uses LLMs/classifiers for semantic understanding
+ Catches subtle/nuanced issues
+ Slower and more expensive

In [ ]:
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

In [ ]:
# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = init_chat_model(model="groq:llama-3.3-70b-versatile", temperature=0)
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.
Input: {text}"""
    result = model.invoke([HumanMessage(content=prompt)])
    if isinstance(result.content, str):
        return result.content.strip()
    return str(result.content).strip()


print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

### Built-in Guardrail (PII Detection Middleware)

Langchain has built-in Middleware for detecting and handling PII (Personally Identifiable Information)

#### Supported PII Types:
| Type | Example |
|---|---|
| `email` | user@example.com |
| `credit_card` | 5105-1051-0510-5100 |
| `ip` | 192.168.1.1 |
| `mac_address` | 00:1A:2B:3C:4D:5E |
| `url` | https://secret-site.com |

#### Strategies:
| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

In [ ]:
from langchain.agents.middleware import PIIMiddleware

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?")]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

In [ ]:
result

### Built-in Guardrail: Human in the Loop Middleware

Pause the agent execution before sensitive operations and waits for human approval. 

Use a `checkpointer` for state persistence across interrupts

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        ),
    ],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig ={"configurable": {"thread_id": "session_001"}}

print("Human-in-the-Loop agent created!")

In [ ]:

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to team@company.com about the Q4 results"
            )
        ]
    },
    config=config,
)

print(" Agent paused — awaiting human approval ")
print(result)

In [ ]:
from langgraph.types import Command

# Human reviews and APPROVES
approved_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config                # Same thread_id to resume the paused session
)

print(" Approved! Final response")
print(approved_result["messages"][-1].content)

In [ ]:
config2: RunnableConfig = {"configurable": {"thread_id": "session_002"}}

agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Delete all records from the users table where active=false"
            )
        ]
    },
    config=config2,
)

rejected_result = agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]
        }
    ),
    config=config2,
)

print(" Rejected! Final response")
print(rejected_result["messages"][-1].content)

### Custom Guardrail using Hooks

**Before-Agent Hook (Input Filter)**

`before_agent()` is used to validate or block requests before any LLM processing begins.

Best for:
1. Keyword/content filtering
2. Authentication checks
3. Rate limiting
4. Blocking specific categories of requests

In [ ]:

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime


class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content_str = cast(str, first_message.content)  # Cast as we know it is going to be str and not dict
        content = content_str.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [
                        AIMessage(
                            content=(
                                "I cannot process requests containing inappropriate content. "
                                "Please rephrase your request."
                            )
                        )
                    ],
                    "jump_to": "end",
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


filtered_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

In [ ]:
result = filtered_agent.invoke(
    {"messages": [HumanMessage(content="What is machine learning?")]}
)
print("Safe request response:")
print(result["messages"][-1].content)

In [ ]:
result = filtered_agent.invoke(
    {"messages": [HumanMessage(content="How do I hack into a server?")]}
)
print("Unsafe request response:")
print(result["messages"][-1].content)

**After-Agent Hook (Output Safety)**

`after_agent()` is used to validate the final agent response before the user sees it.

Best for:
1. Model-based safety evaluation of outputs
2. Compliance scanning (e.g. legal, medical, financial disclaimers)
3. Quality validation
4. Removing sensitive info that slipped through

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime


class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = init_chat_model(
            model="groq:llama-3.3-70b-versatile", temperature=0
        )

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
        content = cast(str, result.content)

        if "UNSAFE" in content.upper():
            print("Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

In [ ]:
result = safe_agent.invoke(
    {"messages": [HumanMessage(content="What is the weather like today?")]}
)
print("Response:")
print(result["messages"][-1].content)

### Layered / Combined Guardrails

Stack multiple guardrails in the middleware array. They execute **in order**, building layered protection.

Input -> **ContentFilterMiddleware** -> **PIIMiddleware**(input) -> **HITLMiddleware** -> **PIIMiddleware**(output) -> **SafetyGuardrailMiddleware** -> Response

In [ ]:
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

production_agent = create_agent(
    model="gpt-4o",
    tools=[search_tool, send_email_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),    # Deterministic input filter
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),         # PII redaction on input

        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}            # Approval for sensitive tools
        ),

        PIIMiddleware("email", strategy="redact", apply_to_output=True),            # PII redaction on output

        SafetyGuardrailMiddleware(),                                                # Model-based output safety
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with 5-layer guardrails created!")

## Real-World Use Case — Healthcare Chatbot

A healthcare chatbot that:

1. Blocks off-topic or harmful requests
2. Redacts patient PII (emails, credit card numbers)
3. Requires human approval before booking appointments
4. Validates that outputs are medically appropriate

In [ ]:
from typing import Any, cast
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.runnables.config import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.types import Command


class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = cast(str, first_msg.content)
        content = content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [
                        AIMessage(
                            content=(
                                "I'm a healthcare assistant and can only help with "
                                "medical questions, appointments, and health information. "
                                "If you're in crisis, please call 112 or your local emergency number."
                            )
                        )
                    ],
                    "jump_to": "end",
                }
        return None

# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n*This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in str(last_message.content).lower():
            last_message.content += self.DISCLAIMER

        return None

# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return (
        f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."
    )

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        HealthcareSafetyFilter(),  # Block harmful/off-topic requests
        PIIMiddleware(
            "email", strategy="redact", apply_to_input=True
        ),  # Redact patient PII from inputs
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        HumanInTheLoopMiddleware(  # Require approval before booking appointments
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),
        MedicalOutputValidator(),  # Add medical disclaimer to all outputs
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    ),
)

print("Healthcare chatbot with full guardrail stack created!")

Safe medical query

In [ ]:
config_t1: RunnableConfig = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [HumanMessage(content="What are symptoms of Type 2 Diabetes?")]},
    config=config_t1
)

result

Query with PII

In [ ]:
result = healthcare_bot.invoke({
    "messages": [HumanMessage(content="My email is patient123@gmail.com. What can I take for a headache?")]},
    config=config_t1
)
print("PII Redaction Test")
print(result["messages"][-1].content)

Harmful request (gets blocked)

In [ ]:
result = healthcare_bot.invoke({
    "messages": [HumanMessage(content="How do I synthesize drugs at home?")]
},
 config=config_t1)
print("Blocked Request")
print(result["messages"][-1].content)

Appointment booking (Human approval required)

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [HumanMessage(content="Book me an appointment with Dr. Sharma on March 15")]},
    config=config
)
print(" Appointment Booking (Awaiting Approval)")
print(result)

# Approve
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n After Approval")
print(approved["messages"][-1].content)